# Hit-and-Run usage notebook

This notebook demonstrates the public `hitandrun` API on a small convex polytope in H-representation (`A x <= b`). It follows a complete workflow:

1. Define a bounded 2D polytope with linear inequalities.
2. Use `MinOver` to find a feasible starting point from an outside guess.
3. Draw reproducible samples with `HitAndRun`.
4. Visualize both the geometry and the sampled marginal distributions.

The figure layout is inspired by the first Font-Clos random-polytopes paper: a constraint-set geometry panel plus smooth marginal-density comparisons. This notebook is a usage demo, not a reproduction of the paper's experiments.

## Setup

Install the package and plotting dependency before running the notebook:

```bash
python -m pip install ".[progress]" matplotlib
```

The sampling cell uses an explicit NumPy random generator so that the figures are reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from hitandrun import HitAndRun, MinOver, Polytope

rng = np.random.default_rng(2026)

plt.rcParams.update({
    "figure.figsize": (7, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

In [ ]:
def polygon_vertices(A, b, tol=1e-9):
    """Return the ordered vertices of a bounded 2D H-polytope."""
    candidates = []
    for i in range(len(b)):
        for j in range(i + 1, len(b)):
            M = A[[i, j]]
            if abs(np.linalg.det(M)) < tol:
                continue
            point = np.linalg.solve(M, b[[i, j]])
            if np.all(A @ point <= b + 1e-8):
                candidates.append(point)

    vertices = np.unique(np.round(candidates, 12), axis=0)
    if vertices.size == 0:
        raise ValueError("No polygon vertices found; check that the polytope is bounded.")

    center = vertices.mean(axis=0)
    angles = np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0])
    return vertices[np.argsort(angles)]


def polygon_area(vertices):
    """Return the area of an ordered 2D polygon."""
    x = vertices[:, 0]
    y = vertices[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def chord_length_at_coordinate(A, b, axis, value, tol=1e-12):
    """Return the feasible chord length at a fixed coordinate value."""
    free_axis = 1 - axis
    lower = -np.inf
    upper = np.inf

    for row, bound in zip(A, b):
        rhs = bound - row[axis] * value
        free_coeff = row[free_axis]
        if abs(free_coeff) < tol:
            if rhs < -tol:
                return 0.0
            continue

        limit = rhs / free_coeff
        if free_coeff > 0:
            upper = min(upper, limit)
        else:
            lower = max(lower, limit)

    if not np.isfinite(lower) or not np.isfinite(upper):
        return 0.0
    return max(0.0, upper - lower)


def theoretical_marginal(A, b, vertices, axis, n_grid=500):
    """Return grid values and exact marginal density for a uniform 2D polytope."""
    grid = np.linspace(vertices[:, axis].min(), vertices[:, axis].max(), n_grid)
    area = polygon_area(vertices)
    lengths = np.array([chord_length_at_coordinate(A, b, axis, value) for value in grid])
    return grid, lengths / area

## Define a polytope in H-representation

The package represents convex polytopes as `A x <= b`. The example below creates a slightly skewed 2D polygon so the geometric constraints and the non-uniform marginal shapes are easy to see.

In [ ]:
A = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
    [0.0, 1.0],
    [0.0, -1.0],
    [1.0, 0.65],
    [-0.45, 1.0],
    [0.55, -1.0],
], dtype=np.float64)

b = np.array([1.25, 1.05, 1.15, 1.00, 1.55, 1.10, 1.05], dtype=np.float64)

polytope = Polytope(A=A, b=b)
vertices = polygon_vertices(A, b)

print(f"dimension: {polytope.dim}")
print(f"constraints: {polytope.nplanes}")
print("ordered vertices:")
print(vertices)

## Find a feasible starting point

`HitAndRun` needs an initial point inside the polytope. `MinOver` can move an outside guess toward feasibility.

In [ ]:
outside_guess = np.array([2.0, 2.0], dtype=np.float64)
interior_point, converged = MinOver(polytope).run(
    speed=0.05,
    starting_point=outside_guess,
    max_iters=1000,
)

assert converged, "MinOver did not find a feasible point."
assert polytope.check_inside(interior_point), "The returned point is outside the polytope."

print(f"outside guess: {outside_guess}")
print(f"interior point: {np.round(interior_point, 4)}")
print(f"largest constraint residual: {np.max(A @ interior_point - b):.4f}")

## Sample uniformly with Hit-and-Run

The sampler repeatedly chooses a random direction, intersects that line with the constraint planes, and moves to a uniformly selected point along the feasible chord.

In [ ]:
sampler = HitAndRun(polytope=polytope, starting_point=interior_point, rng=rng)
raw_samples = sampler.get_samples(n_samples=5_000, thin=2)
samples = raw_samples[500:]  # drop a short burn-in for plotting

constraint_residuals = samples @ A.T - b
assert np.all(constraint_residuals <= 1e-10), "At least one sample left the polytope."

print(f"kept samples: {samples.shape[0]}")
print(f"sample mean: {np.round(samples.mean(axis=0), 4)}")
print(f"max constraint residual: {constraint_residuals.max():.2e}")

## Geometry view

The first figure overlays the feasible polygon, an outside MinOver starting guess, the feasible point it finds, and the first Hit-and-Run moves.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.2))
closed_vertices = np.vstack([vertices, vertices[0]])

ax.fill(closed_vertices[:, 0], closed_vertices[:, 1], color="#dce8f2", alpha=0.9, label="feasible polytope")
ax.plot(closed_vertices[:, 0], closed_vertices[:, 1], color="#234b6d", lw=2)
ax.scatter(samples[::20, 0], samples[::20, 1], s=8, alpha=0.22, color="#f05a28", label="samples")

trajectory = np.vstack([interior_point, raw_samples[:160]])
segments = np.stack([trajectory[:-1], trajectory[1:]], axis=1)
line_collection = LineCollection(segments, colors="#333333", linewidths=0.8, alpha=0.45)
ax.add_collection(line_collection)

ax.scatter(*outside_guess, marker="x", s=90, color="#7f7f7f", label="outside guess")
ax.scatter(*interior_point, marker="o", s=70, color="#111111", label="MinOver point")

ax.set_title("Hit-and-Run inside a constrained polygon")
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.set_aspect("equal", adjustable="box")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
plt.show()

## Marginal distributions

The paper-style diagnostic is to compare entire marginal profiles, not only point summaries. For a uniform distribution on a 2D polytope, the exact marginal density at a coordinate value is the feasible chord length at that value divided by the polygon area. The sampled histograms should track these theoretical profiles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
colors = ["#2563a5", "#cc4c2f"]
labels = ["x0", "x1"]

for axis_index, axis, color, label, values in zip(range(2), axes, colors, labels, samples.T):
    density, edges = np.histogram(values, bins=55, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    theory_grid, theory_density = theoretical_marginal(A, b, vertices, axis_index)
    theory_integral = np.trapezoid(theory_density, theory_grid)
    assert np.isclose(theory_integral, 1.0, atol=5e-3)

    axis.fill_between(centers, density, step="mid", alpha=0.30, color=color)
    axis.plot(centers, density, color=color, lw=1.6, alpha=0.85, label=f"sampled p({label})")
    axis.plot(theory_grid, theory_density, color="#222222", lw=2.2, label="theoretical")
    axis.axvline(values.mean(), color="#222222", ls="--", lw=1, alpha=0.75, label="sample mean")
    axis.set_xlabel(label)
    axis.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=1)

axes[0].set_ylabel("density")
fig.suptitle("Hit-and-Run marginals against exact polytope profiles", y=1.02)
fig.tight_layout()
plt.show()

## Next steps

Try changing `A` and `b` to describe a different bounded polytope, increasing `thin` to reduce serial correlation, or replacing `rng` with a different seed to generate an independent reproducible run.